In [0]:
tables = ["orders", "customers", "order_items", "order_payments", "products", "sellers"]

print(f"{'Table':<20}{'Bronze':<12}{'Silver':<12}")
for t in tables:
    bronze_count = spark.table(f"olist_project.bronze.{t}").count()
    silver_count = spark.table(f"olist_project.silver.{t}").count()
    print(f"{t:<20}{bronze_count:<12}{silver_count:<12}")

Table               Bronze      Silver      
orders              99441       99441       
customers           99441       99441       
order_items         112650      112650      
order_payments      103886      103886      
products            32951       32951       
sellers             3095        3095        


In [0]:
from pyspark.sql.functions import col, count, when

def null_check(table, key_cols):
    df = spark.table(table)
    print(f"\n{table}:")
    for c in key_cols:
        nulls = df.filter(col(c).isNull()).count()
        print(f"  {c}: {nulls} nulls")

null_check("olist_project.silver.orders", ["order_id", "customer_id"])
null_check("olist_project.silver.customers", ["customer_id"])
null_check("olist_project.silver.order_items", ["order_id", "product_id"])
null_check("olist_project.gold.fact_orders", ["order_id", "customer_id"])


olist_project.silver.orders:
  order_id: 0 nulls
  customer_id: 0 nulls

olist_project.silver.customers:
  customer_id: 0 nulls

olist_project.silver.order_items:
  order_id: 0 nulls
  product_id: 0 nulls

olist_project.gold.fact_orders:
  order_id: 0 nulls
  customer_id: 0 nulls


In [0]:
def duplicate_check(table, key_col):
    df = spark.table(table)
    total = df.count()
    distinct = df.select(key_col).distinct().count()
    print(f"{table}: {total} rows, {distinct} distinct {key_col} ({'OK' if total == distinct else 'DUPLICATES FOUND'})")

duplicate_check("olist_project.silver.orders", "order_id")
duplicate_check("olist_project.silver.customers", "customer_id")
duplicate_check("olist_project.gold.fact_orders", "order_id")

olist_project.silver.orders: 99441 rows, 99441 distinct order_id (OK)
olist_project.silver.customers: 99441 rows, 99441 distinct customer_id (OK)
olist_project.gold.fact_orders: 99441 rows, 99441 distinct order_id (OK)


In [0]:
fact = spark.table("olist_project.gold.fact_orders")
dim_customer = spark.table("olist_project.gold.dim_customer")

orphans = fact.join(dim_customer, "customer_id", "left_anti")
print(f"Orders with no matching customer: {orphans.count()}")

Orders with no matching customer: 0


In [0]:
checks_passed = True  # set this based on what you saw above — be honest if something failed

print("=" * 40)
print("DATA QUALITY SUMMARY")
print("=" * 40)
print(f"Overall status: {'PASS' if checks_passed else 'FAIL - see details above'}")

DATA QUALITY SUMMARY
Overall status: PASS


In [0]:
from pyspark.sql.functions import col, count, when, isnan

def null_audit(table_name):
    df = spark.table(table_name)
    total = df.count()
    print(f"\n{'='*70}")
    print(f"NULL AUDIT: {table_name} ({total} rows)")
    print(f"{'='*70}")
    for c, dtype in df.dtypes:
        if dtype in ("double", "float"):
            null_count = df.filter(col(c).isNull() | isnan(col(c))).count()
        else:
            null_count = df.filter(col(c).isNull()).count()
        pct = round((null_count / total) * 100, 2) if total > 0 else 0
        if null_count > 0:
            print(f"  {c:<30} {null_count:>8} nulls  ({pct}%)")

null_audit("olist_project.silver.orders")
null_audit("olist_project.silver.customers")
null_audit("olist_project.silver.order_items")
null_audit("olist_project.silver.products")
null_audit("olist_project.gold.fact_orders")


NULL AUDIT: olist_project.silver.orders (99441 rows)
  approved_ts                         160 nulls  (0.16%)
  delivered_carrier_ts               1783 nulls  (1.79%)
  delivered_customer_ts              2965 nulls  (2.98%)

NULL AUDIT: olist_project.silver.customers (99441 rows)

NULL AUDIT: olist_project.silver.order_items (112650 rows)

NULL AUDIT: olist_project.silver.products (32951 rows)
  product_category_name               610 nulls  (1.85%)
  product_name_lenght                 610 nulls  (1.85%)
  product_description_lenght          610 nulls  (1.85%)
  product_photos_qty                  610 nulls  (1.85%)
  product_weight_g                      2 nulls  (0.01%)
  product_length_cm                     2 nulls  (0.01%)
  product_height_cm                     2 nulls  (0.01%)
  product_width_cm                      2 nulls  (0.01%)

NULL AUDIT: olist_project.gold.fact_orders (99441 rows)
  delivered_customer_ts              2965 nulls  (2.98%)
  total_item_price              